In [16]:
import pandas as pd
import joblib as jb
import numpy as np

In [8]:
url="../data/Telco-Customer-Churn.csv"
df= pd.read_csv(url)
RANDOM_STATE=66

In [13]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,GridSearchCV,learning_curve

In [5]:
def prepare_data(df: pd.DataFrame):
    df = df.copy()

    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
    
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
    
    X = df.drop(columns=['customerID', 'Churn'])
    y = df['Churn']
    
    return X, y

In [ ]:
def build_preprocessor(X: pd.DataFrame):
    num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_cols = X.select_dtypes(include=['str']).columns.tolist()  

    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    return ColumnTransformer(transformers=[
        ('nums', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ])

In [28]:
models_and_grid = {
    'Decision Tree': {
        "model": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
        "params": {
            'classifier__criterion': ['gini', 'entropy'],
            'classifier__max_depth': [3, 5, 8, 12],
            'classifier__min_samples_split': [10, 20, 50],
            'classifier__min_samples_leaf': [5, 10, 20]  
        }
    },
    'SVM': {
        "model": SVC(random_state=RANDOM_STATE, class_weight='balanced'),
        "params": {
            'classifier__C': [0.01, 0.1, 1, 10],
            'classifier__kernel': ['linear', 'rbf'],
            'classifier__gamma': ['scale', 'auto', 0.01]
        }
    },
    'Random Forest': {
        "model": RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
        "params": {
            'classifier__n_estimators': [100, 200],
            'classifier__max_depth': [4, 6, 10],         
            'classifier__min_samples_split': [20, 50],    
            'classifier__min_samples_leaf': [10, 20],     
            'classifier__max_features': ['sqrt']
        }
    }
}

models_and_gridv2 = {
    'Decision Tree': {
        "model": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
        "params": {
            "classifier__max_depth": [3, 5, 10, None],
            "classifier__min_samples_split": [2, 10, 20],
            "classifier__min_samples_leaf": [1, 5, 10]
        }
    },
    'Random Forest': {
        "model": RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
        "params": {
            "classifier__n_estimators": [100, 200, 300],
            "classifier__max_depth": [5, 10, 20, None],
            "classifier__min_samples_split": [2, 10],
            "classifier__min_samples_leaf": [1, 5],
            "classifier__max_features": ["sqrt", 0.5]
        }
    },
    'SVM': {
        "model": SVC(random_state=RANDOM_STATE, class_weight='balanced'),
        "params": {
            "classifier__C": [0.1, 1, 10, 100],
            "classifier__gamma": ["scale", "auto", 0.01, 0.1],
            "classifier__kernel": ["rbf"]
        }
    }
}
models_and_gridv3 = {
    'Decision Tree': {
        "model": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "params": {
            "classifier__criterion": ["gini", "entropy", "log_loss"],
            "classifier__max_depth": [3, 5, 8, 12, 15, None],
            "classifier__min_samples_split": [2, 5, 10, 20],
            "classifier__min_samples_leaf": [1, 2, 4, 8],
            "classifier__class_weight": ["balanced", {0: 1, 1: 1.5}, {0: 1, 1: 2}]
        }
    },
    'Random Forest': {
        "model": RandomForestClassifier(random_state=RANDOM_STATE),
        "params": {
            "classifier__n_estimators": [100, 200, 300, 400],
            "classifier__criterion": ["gini", "entropy"],
            "classifier__max_depth": [5, 10, 15, 25, None],
            "classifier__min_samples_split": [2, 5, 10],
            "classifier__min_samples_leaf": [1, 2, 4],
            "classifier__max_features": ["sqrt", "log2", 0.4, 0.6],
            "classifier__class_weight": ["balanced", "balanced_subsample", {0: 1, 1: 2}]
        }
    },
    'SVM': {
        "model": SVC(random_state=RANDOM_STATE),
        "params": {
            "classifier__C": [0.1, 1, 10, 50, 100],
            "classifier__gamma": ["scale", "auto", 0.001, 0.01, 0.1],
            "classifier__kernel": ["rbf", "poly"],
            "classifier__degree": [2, 3], # Used if kernel is 'poly'
            "classifier__class_weight": ["balanced", {0: 1, 1: 2}, {0: 1, 1: 3}]
        }
    }
}

In [30]:

X,y=prepare_data(df)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y)
preprocessor = build_preprocessor(X_train)

best_recall_score = 0
champion_model = None
champion_name = ""
grid_results = {}

for name, config in models_and_gridv2.items():
    print(f"\n==============================")
    print(f"Tuning {name} (Target: Recall)")
    print(f"==============================")
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', config['model'])
    ])
    
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=config['params'],
        cv=5,
        scoring='recall',         
        return_train_score=True,
        n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train)
    
    best_idx = grid_search.best_index_
    train_recall = grid_search.cv_results_['mean_train_score'][best_idx]
    val_recall = grid_search.cv_results_['mean_test_score'][best_idx]
    
    print(f"Best CV Validation Recall: {val_recall:.4f}")
    print(f"Corresponding Train Recall: {train_recall:.4f}")
    
    # Check variance gap
    if train_recall - val_recall > 0.08:
        print(f"⚠️ Warning: {name} still shows variance (overfitting).")
    else:
        print(f"✅ {name} has a well-controlled variance profile.")

    grid_results[name] = {
        'model': grid_search.best_estimator_,
        'params': grid_search.best_params_,
        'val_recall': val_recall
    }
    
    if val_recall > best_recall_score:
        best_recall_score = val_recall
        champion_model = grid_search.best_estimator_
        champion_name = name

print(f"\n🏆 Recall Champion: {champion_name} with Validation Recall: {best_recall_score:.4f}")

model_path = '../models/churn_v3.joblib'
jb.dump(champion_model, model_path)
print(f"\n💾 Champion model ({champion_name}) successfully saved to: {model_path}")


Tuning Decision Tree (Target: Recall)


Best CV Validation Recall: 0.8000
Corresponding Train Recall: 0.8349
✅ Decision Tree has a well-controlled variance profile.

Tuning Random Forest (Target: Recall)
Best CV Validation Recall: 0.8074
Corresponding Train Recall: 0.8304
✅ Random Forest has a well-controlled variance profile.

Tuning SVM (Target: Recall)
Best CV Validation Recall: 0.8020
Corresponding Train Recall: 0.8099
✅ SVM has a well-controlled variance profile.

🏆 Recall Champion: Random Forest with Validation Recall: 0.8074

💾 Champion model (Random Forest) successfully saved to: ../models/churn_v3.joblib


In [31]:
def plot_or_check_learning_curve(estimator, X, y):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, cv=5, scoring='accuracy',
        train_sizes=np.linspace(0.2, 1.0, 5), n_jobs=-1
    )
    
    print("\n--- Learning Curve Diagnostic ---")
    for size, t_scores, v_scores in zip(train_sizes, train_scores, val_scores):
        print(f"Training Size: {size:4.0f} | Train Mean: {t_scores.mean():.3f} | Validation Mean: {v_scores.mean():.3f}")

plot_or_check_learning_curve(champion_model, X_train, y_train)


--- Learning Curve Diagnostic ---
Training Size:  901 | Train Mean: 0.788 | Validation Mean: 0.741
Training Size: 1802 | Train Mean: 0.768 | Validation Mean: 0.739
Training Size: 2704 | Train Mean: 0.762 | Validation Mean: 0.739
Training Size: 3605 | Train Mean: 0.753 | Validation Mean: 0.736
Training Size: 4507 | Train Mean: 0.745 | Validation Mean: 0.733


In [32]:
from sklearn.metrics import classification_report, confusion_matrix, recall_score

model_path = '../models/churn_v3.joblib'
jb.dump(champion_model, model_path)
print(f"\n💾 Champion model locked and saved to: {model_path}")

y_pred = champion_model.predict(X_test)

final_test_recall = recall_score(y_test, y_pred)
print(f"\n🎯 Final Test Set Recall: {final_test_recall:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


💾 Champion model locked and saved to: ../models/churn_v3.joblib

🎯 Final Test Set Recall: 0.8529

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.72      0.81      1035
           1       0.52      0.85      0.65       374

    accuracy                           0.76      1409
   macro avg       0.73      0.79      0.73      1409
weighted avg       0.82      0.76      0.77      1409


Confusion Matrix:
[[745 290]
 [ 55 319]]
